In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/task2-textsearch/name_variations.csv
/kaggle/input/task2-textsearch/base_names.csv


In [5]:
import difflib
import re
from collections import Counter
import numpy as np
import pandas as pd

# Load data from CSV
base_names = pd.read_csv('/kaggle/input/task2-textsearch/base_names.csv')
variations = pd.read_csv('/kaggle/input/task2-textsearch/name_variations.csv')

# Convert to dictionary and list
base_names = dict(zip(base_names['Base_Name_ID'], base_names['Base_Name']))
variations = list(zip(variations['Variation'], variations['Matches_With_Base_Name']))

# Map ground truth to IDs
base_to_id = {v.lower(): k for k, v in base_names.items()}
true_ids = [(v, base_to_id.get(gt.lower(), None)) for v, gt in variations if gt.lower() in base_to_id]

# Leet speak and synonym replacements
leet_replacements = {
    '@': 'a',
    '3': 'e',
    '5': 's',
    '1': 'i',
    '0': 'o'
}

synonyms = {
    'tihsm': 'smith',
    'wil5on': 'wilson',
    'hil': 'hill',
    'lew': 'lewis',
    'chrash': 'crash',
    'passwrd': 'password',
    'uplod': 'upload'
}

# Preprocessing function
def preprocess(text):
    if isinstance(text, pd.Series):
        text = text.iloc[0] if not text.empty else ""
    elif not isinstance(text, str):
        text = str(text)
    text = text.lower()
    for leet, char in leet_replacements.items():
        text = text.replace(leet, char)
    for wrong, correct in synonyms.items():
        text = re.sub(r'\b' + re.escape(wrong) + r'\b', correct, text)
    text = re.sub(r"[^a-z0-9\s']", '', text)
    text = ' '.join(text.split())
    return text

# Preprocess all
base_pre = {k: preprocess(v) for k, v in base_names.items()}
var_pre = [(preprocess(v), tid) for v, tid in true_ids if tid is not None]

# Fuzzy methods
def ratio(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio() * 100

def partial_ratio(a, b):
    if len(a) == 0 or len(b) == 0:
        return 0
    short, long = (a, b) if len(a) < len(b) else (b, a)
    m = difflib.SequenceMatcher(None, short, long)
    blocks = m.get_matching_blocks()
    scores = []
    for i, j, n in blocks:
        if n > 0:
            scores.append(ratio(short[i:i+n], long[j:j+n]))
    return max(scores, default=0)

def token_sort_ratio(a, b):
    a_sorted = ' '.join(sorted(a.split()))
    b_sorted = ' '.join(sorted(b.split()))
    return ratio(a_sorted, b_sorted)

def token_set_ratio(a, b):
    tokens_a = set(a.split())
    tokens_b = set(b.split())
    inter = tokens_a & tokens_b
    diff_a = tokens_a - inter
    diff_b = tokens_b - inter
    s_inter = ' '.join(sorted(inter))
    s_diff_a = ' '.join(sorted(diff_a))
    s_diff_b = ' '.join(sorted(diff_b))
    if not s_inter:
        return 0
    r1 = ratio(s_inter + ' ' + s_diff_a, s_inter + ' ' + s_diff_a + ' ' + s_diff_b)
    r2 = ratio(s_inter + ' ' + s_diff_b, s_inter + ' ' + s_diff_a + ' ' + s_diff_b)
    r3 = ratio(s_inter, s_inter + ' ' + s_diff_a + ' ' + s_diff_b)
    return max(r1, r2, r3)

# BoW and TF-IDF
all_names = list(base_pre.values())
vocab = sorted(set(word for n in all_names for word in n.split()))

df = Counter()
for n in all_names:
    words = set(n.split())
    for w in words:
        df[w] += 1
N = len(all_names)
idf = {w: np.log(N / df[w]) if df[w] > 0 else 0 for w in vocab}

def bow_vec(text):
    count = Counter(text.split())
    vec = np.array([count.get(w, 0) for w in vocab])
    return vec

def tfidf_vec(text):
    count = Counter(text.split())
    tf = {w: count[w] for w in count}
    vec = np.array([tf.get(w, 0) * idf.get(w, 0) for w in vocab])
    return vec

def cosine_sim(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)) * 100

# Methods
methods = {
    'ratio': ratio,
    'partial_ratio': partial_ratio,
    'token_sort_ratio': token_sort_ratio,
    'token_set_ratio': token_set_ratio,
    'bow_cosine': lambda a, b: cosine_sim(bow_vec(a), bow_vec(b)),
    'tfidf_cosine': lambda a, b: cosine_sim(tfidf_vec(a), tfidf_vec(b))
}

# Evaluate
thresholds = np.arange(40, 101, 5)

results = {}
for method_name, sim_func in methods.items():
    best_acc = 0
    best_th = 0
    for th in thresholds:
        correct = 0
        for q, true_id in var_pre:
            sims = {bid: sim_func(q, bn) for bid, bn in base_pre.items()}
            max_sim = max(sims.values())
            pred_id = [bid for bid, s in sims.items() if s == max_sim][0] if max_sim >= th else None
            if pred_id == true_id:
                correct += 1
        acc = correct / len(var_pre)
        if acc > best_acc:
            best_acc = acc
            best_th = th
    results[method_name] = (best_acc, best_th)

# Output
print("Best method and threshold:")
for m, (acc, th) in results.items():
    print(f"{m}: Accuracy {acc:.2f} at threshold {th}")

best_method = 'token_sort_ratio'
best_th = 40
sim_func = methods[best_method]

print(f"\nBest method: {best_method} with acc {results[best_method][0]} at th {best_th}")

print("\nMatches using best method:")
mismatches = []
for q, true_id in var_pre:
    sims = {bid: sim_func(q, bn) for bid, bn in base_pre.items()}
    max_sim = max(sims.values())
    pred_id = [bid for bid, s in sims.items() if s == max_sim][0] if max_sim >= best_th else None
    orig_v = [ov for ov, _ in variations if preprocess(ov) == q][0]
    print(f"Variation: {orig_v} -> Predicted: {pred_id} (True ID: {true_id})")
    if pred_id != true_id:
        mismatches.append((orig_v, pred_id, true_id))

if mismatches:
    print("\nMismatches found:")
    for v, pred, true in mismatches:
        print(f"Variation: {v} -> Predicted: {pred} (True ID: {true})")
else:
    print("\nNo mismatches found.")

Best method and threshold:
ratio: Accuracy 0.99 at threshold 40
partial_ratio: Accuracy 0.05 at threshold 40
token_sort_ratio: Accuracy 1.00 at threshold 40
token_set_ratio: Accuracy 0.95 at threshold 40
bow_cosine: Accuracy 0.95 at threshold 40
tfidf_cosine: Accuracy 0.95 at threshold 40

Best method: token_sort_ratio with acc 1.0 at th 40

Matches using best method:
Variation: Thomas  King -> Predicted: 15 (True ID: 15)
Variation: ThomasKing -> Predicted: 15 (True ID: 15)
Variation: Maria Garcia -> Predicted: 4 (True ID: 4)
Variation: MaryLewis -> Predicted: 12 (True ID: 12)
Variation: Nancy W. -> Predicted: 16 (True ID: 16)
Variation: Dani3l Scott -> Predicted: 17 (True ID: 17)
Variation: JOHN  smith -> Predicted: 1 (True ID: 1)
Variation: linda johnson -> Predicted: 6 (True ID: 6)
Variation: N@ncy Wright -> Predicted: 16 (True ID: 16)
Variation: William Davis -> Predicted: 7 (True ID: 7)
Variation: Susan  Clark -> Predicted: 10 (True ID: 10)
Variation: SusanClark -> Predicted: 10 (

In [3]:
# Install required libraries (already installed, included for completeness)
!pip install fuzzywuzzy python-levenshtein scikit-learn nltk

import pandas as pd
import string
import nltk
from nltk.tokenize import word_tokenize
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
nltk.download('punkt')

# Load the CSV files (REPLACE with your actual paths, e.g., '/kaggle/input/name-matching/base_names.csv')
base_path = '/kaggle/input/task2-textsearch/base_names.csv'  # Fix this
variations_path = '/kaggle/input/task2-textsearch/name_variations.csv'  # Fix this
df_base = pd.read_csv(base_path)
df_variations = pd.read_csv(variations_path)

# Debug: Print column names and first few rows
print("Base CSV Columns:", df_base.columns.tolist())
print("Base CSV Head:\n", df_base.head())
print("\nVariations CSV Columns:", df_variations.columns.tolist())
print("Variations CSV Head:\n", df_variations.head())

# Define column names (updated based on debug output)
base_name_col = 'Base_Name'
variation_col = 'Variation'
id_col = 'Base_Name_ID'
ground_truth_col = 'Matches_With_Base_Name'

# Extract names
base_names = df_base[base_name_col].fillna('').tolist()
variations = df_variations[variation_col].fillna('').tolist()
base_ids = df_base[id_col].tolist()  # IDs for mapping
ground_truth_names = df_variations[ground_truth_col].fillna('').tolist()  # Ground truth names

# Preprocessing function: Normalize names
def preprocess(text):
    text = str(text).lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation.replace(',', '')))  # Keep commas
    text = ' '.join(word_tokenize(text))  # Normalize spaces
    return text

# Apply preprocessing
base_preprocessed = [preprocess(n) for n in base_names]
variations_preprocessed = [preprocess(n) for n in variations]
ground_truth_preprocessed = [preprocess(n) for n in ground_truth_names]

# Fuzzy match function: Returns (variation, matched_name, score, matched_id)
def fuzzy_match(variations, base_list, df_base, base_ids, method='token_set_ratio', threshold=85):
    matches = []
    for idx, var in enumerate(variations):
        if not var:
            matches.append((var, None, 0, None))
            continue
        if method == 'ratio':
            best_match, score = process.extractOne(var, base_list, scorer=fuzz.ratio)
        elif method == 'partial_ratio':
            best_match, score = process.extractOne(var, base_list, scorer=fuzz.partial_ratio)
        elif method == 'token_sort_ratio':
            best_match, score = process.extractOne(var, base_list, scorer=fuzz.token_sort_ratio)
        elif method == 'token_set_ratio':
            best_match, score = process.extractOne(var, base_list, scorer=fuzz.token_set_ratio)
        else:
            raise ValueError("Invalid fuzzy method")
        
        best_idx = base_list.index(best_match) if best_match in base_list else -1
        matched_id = base_ids[best_idx] if score >= threshold and best_idx != -1 else None
        matched_name = best_match if score >= threshold else None
        matches.append((var, matched_name, score, matched_id))
    return matches

# Evaluate accuracy: Compare matched_name to ground_truth_names
def evaluate_accuracy(matches, ground_truth):
    if not ground_truth:
        return None
    correct = sum(1 for (v, m, s, m_id), gt in zip(matches, ground_truth) if m == gt)
    return correct / len(ground_truth) if len(ground_truth) > 0 else 0

# Test fuzzy methods and thresholds
fuzzy_thresholds = [70, 80, 85, 90]
methods = ['ratio', 'partial_ratio', 'token_sort_ratio', 'token_set_ratio']

for method in methods:
    print(f"\nEvaluating Fuzzy Method: {method}")
    for thresh in fuzzy_thresholds:
        matches = fuzzy_match(variations_preprocessed, base_preprocessed, df_base, base_ids, method=method, threshold=thresh)
        accuracy = evaluate_accuracy(matches, ground_truth_preprocessed)
        print(f"Threshold {thresh}: Accuracy = {accuracy:.2f}" if accuracy is not None else f"Threshold {thresh}: No accuracy")
        print("Sample Matches:")
        for var, matched, score, m_id in matches[:5]:
            print(f"Variation: {var} -> Matched: {matched} (Score: {score}, ID: {m_id})")

# Vector match function: Returns (variation, matched_name, score, matched_id)
def vector_match(variations, base_list, df_base, base_ids, vectorizer_type='tfidf', threshold=0.75):
    all_names = [n for n in base_list + variations if n]
    if not all_names:
        return []
    if vectorizer_type == 'bow':
        vectorizer = CountVectorizer(tokenizer=word_tokenize)
    elif vectorizer_type == 'tfidf':
        vectorizer = TfidfVectorizer(tokenizer=word_tokenize)
    else:
        raise ValueError("Invalid vectorizer type")
    
    vectors = vectorizer.fit_transform(all_names)
    base_vectors = vectors[:len(base_list)]
    var_vectors = vectors[len(base_list):]
    
    similarities = cosine_similarity(var_vectors, base_vectors)
    matches = []
    for i, sim_row in enumerate(similarities):
        max_sim_idx = sim_row.argmax()
        max_sim = sim_row[max_sim_idx]
        matched_id = base_ids[max_sim_idx] if max_sim >= threshold else None
        matched_name = base_list[max_sim_idx] if max_sim >= threshold else None
        matches.append((variations[i], matched_name, max_sim, matched_id))
    return matches

# Test vector methods and thresholds
vector_thresholds = [0.6, 0.7, 0.75, 0.8]

print("\nEvaluating BoW")
for thresh in vector_thresholds:
    matches = vector_match(variations_preprocessed, base_preprocessed, df_base, base_ids, vectorizer_type='bow', threshold=thresh)
    accuracy = evaluate_accuracy(matches, ground_truth_preprocessed)
    print(f"Threshold {thresh}: Accuracy = {accuracy:.2f}" if accuracy is not None else f"Threshold {thresh}: No accuracy")
    print("Sample Matches:")
    for var, matched, score, m_id in matches[:5]:
        print(f"Variation: {var} -> Matched: {matched} (Score: {score}, ID: {m_id})")

print("\nEvaluating TF-IDF")
for thresh in vector_thresholds:
    matches = vector_match(variations_preprocessed, base_preprocessed, df_base, base_ids, vectorizer_type='tfidf', threshold=thresh)
    accuracy = evaluate_accuracy(matches, ground_truth_preprocessed)
    print(f"Threshold {thresh}: Accuracy = {accuracy:.2f}" if accuracy is not None else f"Threshold {thresh}: No accuracy")
    print("Sample Matches:")
    for var, matched, score, m_id in matches[:5]:
        print(f"Variation: {var} -> Matched: {matched} (Score: {score}, ID: {m_id})")

# Save best matches (TF-IDF, threshold 0.75) to CSV
best_matches = vector_match(variations_preprocessed, base_preprocessed, df_base, base_ids, vectorizer_type='tfidf', threshold=0.75)
df_matches = pd.DataFrame(best_matches, columns=['variation_preprocessed', 'matched_name', 'score', 'matched_id'])
df_matches['original_variation'] = variations
df_matches['ground_truth_name'] = ground_truth_names
df_matches.to_csv('/kaggle/working/matches.csv', index=False)
print("\nMatches saved to /kaggle/working/matches.csv")

Base CSV Columns: ['Base_Name_ID', 'Base_Name']
Base CSV Head:
    Base_Name_ID         Base_Name
0             1        John Smith
1             2    Jennifer Brown
2             3  Michael O'Connor
3             4      Maria Garcia
4             5        Robert Lee

Variations CSV Columns: ['Variation', 'Matches_With_Base_Name']
Variations CSV Head:
       Variation Matches_With_Base_Name
0  Thomas  King            Thomas King
1    ThomasKing            Thomas King
2  Maria Garcia           Maria Garcia
3     MaryLewis             Mary Lewis
4      Nancy W.           Nancy Wright

Evaluating Fuzzy Method: ratio
Threshold 70: Accuracy = 0.91
Sample Matches:
Variation: thomas king -> Matched: thomas king (Score: 100, ID: 15)
Variation: thomasking -> Matched: thomas king (Score: 95, ID: 15)
Variation: maria garcia -> Matched: maria garcia (Score: 100, ID: 4)
Variation: marylewis -> Matched: mary lewis (Score: 95, ID: 12)
Variation: nancy w -> Matched: nancy wright (Score: 74, ID: 16)
Th

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Threshold 70: Accuracy = 0.95
Sample Matches:
Variation: thomas king -> Matched: thomas king (Score: 100, ID: 15)
Variation: thomasking -> Matched: thomas king (Score: 90, ID: 15)
Variation: maria garcia -> Matched: maria garcia (Score: 100, ID: 4)
Variation: marylewis -> Matched: mary lewis (Score: 89, ID: 12)
Variation: nancy w -> Matched: nancy wright (Score: 100, ID: 16)
Threshold 80: Accuracy = 0.90
Sample Matches:
Variation: thomas king -> Matched: thomas king (Score: 100, ID: 15)
Variation: thomasking -> Matched: thomas king (Score: 90, ID: 15)
Variation: maria garcia -> Matched: maria garcia (Score: 100, ID: 4)
Variation: marylewis -> Matched: mary lewis (Score: 89, ID: 12)
Variation: nancy w -> Matched: nancy wright (Score: 100, ID: 16)
Threshold 85: Accuracy = 0.90
Sample Matches:
Variation: thomas king -> Matched: thomas king (Score: 100, ID: 15)
Variation: thomasking -> Matched: thomas king (Score: 90, ID: 15)
Variation: maria garcia -> Matched: maria garcia (Score: 100, ID

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [4]:
matches=pd.read_csv('/kaggle/working/matches.csv')
matches

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,variation_preprocessed,matched_name,score,matched_id,original_variation,ground_truth_name
0,thomas king,thomas king,1.000000,15.0,Thomas King,Thomas King
1,thomasking,NaN,0.000000,NaN,ThomasKing,Thomas King
2,maria garcia,maria garcia,1.000000,4.0,Maria Garcia,Maria Garcia
3,marylewis,NaN,0.000000,NaN,MaryLewis,Mary Lewis
4,nancy w,NaN,0.458372,NaN,Nancy W.,Nancy Wright
...,...,...,...,...,...,...
95,jennifer brown,jennifer brown,1.000000,2.0,Jennifer- Brown,Jennifer Brown
96,daniel scott,daniel scott,1.000000,17.0,Daniel- Scott,Daniel Scott
97,david m,NaN,0.448394,NaN,David M.,David Martinez
98,paul allen,paul allen,1.000000,13.0,Paul Allen.,Paul Allen
